In [42]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import re

In [43]:
output_path_xa = (r"../../data/raw/density_xa.csv")
output_path_phuong = (r"../../data/raw/density_phuong.csv")
output_full_density =  (r"../../data/processed/density.csv")

In [44]:
# URL để lấy DENSITY của xã 
url = 'https://vi.wikipedia.org/wiki/Danh_s%C3%A1ch_x%C3%A3_t%E1%BA%A1i_Vi%E1%BB%87t_Nam'

# Gửi yêu cầu HTTP với header giống trình duyệt để tránh bị chặn
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
}
response = requests.get(url, headers=headers, timeout=30)
print('HTTP status:', response.status_code)

if response.status_code != 200:
    raise Exception(f'Failed to fetch the page: {response.status_code}')

# Parse HTML -> soup
soup = BeautifulSoup(response.text, "html.parser")

# Lấy bảng chính trên trang và chuyển thành DataFrame bằng BeautifulSoup
table = soup.select_one('table.wikitable.sortable') or soup.find('table')
if table is None:
    raise Exception('Không tìm thấy bảng nào trên trang.')

rows = []
for tr in table.find_all('tr'):
    cells = [cell.get_text(' ', strip=True) for cell in tr.find_all(['th', 'td'])]
    if cells:
        rows.append(cells)

if len(rows) < 2:
    raise Exception('Bảng không có đủ dữ liệu để chuyển thành DataFrame.')

density_xa = pd.DataFrame(rows[1:], columns=rows[0])

# Làm sạch các ký hiệu chú thích kiểu [1], [2] trong dữ liệu
density_xa = density_xa.replace(r'\s*\[\d+\]', '', regex=True)

print('Shape:', density_xa.shape)
print(density_xa.head())

HTTP status: 200
Shape: (2611, 7)
  STT       Tên Thuộc tỉnh/thành phố Diện tích (km²) Dân số năm 2025 (người)  \
0   1     A Dơi            Quảng Trị          112,41                  11.058   
1   2  A Lưới 1                  Huế          198,59                  12.403   
2   3  A Lưới 2                  Huế           97,62                  20.496   
3   4  A Lưới 3                  Huế          154,23                   8.976   
4   5  A Lưới 4                  Huế          233,66                  10.752   

  Mật độ dân số (người/km²) Năm thành lập  
0                        98          2025  
1                        62          2025  
2                       210          2025  
3                        58          2025  
4                        46          2025  


In [45]:
# URL để lấy DENSITY của phường
url = 'https://vi.wikipedia.org/wiki/Danh_s%C3%A1ch_ph%C6%B0%E1%BB%9Dng_t%E1%BA%A1i_Vi%E1%BB%87t_Nam'

# Gửi yêu cầu HTTP với header giống trình duyệt để tránh bị chặn
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36',
    'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7',
}
response = requests.get(url, headers=headers, timeout=30)
print('HTTP status:', response.status_code)

if response.status_code != 200:
    raise Exception(f'Failed to fetch the page: {response.status_code}')

# Parse HTML -> soup
soup = BeautifulSoup(response.text, "html.parser")

# Lấy bảng chính trên trang và chuyển thành DataFrame bằng BeautifulSoup
table = soup.select_one('table.wikitable.sortable') or soup.find('table')
if table is None:
    raise Exception('Không tìm thấy bảng nào trên trang.')

rows = []
for tr in table.find_all('tr'):
    cells = [cell.get_text(' ', strip=True) for cell in tr.find_all(['th', 'td'])]
    if cells:
        rows.append(cells)

if len(rows) < 2:
    raise Exception('Bảng không có đủ dữ liệu để chuyển thành DataFrame.')

density_phuong = pd.DataFrame(rows[1:], columns=rows[0])

# Làm sạch các ký hiệu chú thích kiểu [1], [2] trong dữ liệu
density_phuong = density_phuong.replace(r'\s*\[\d+\]', '', regex=True)

print('Shape:', density_phuong.shape)
print(density_phuong.head())

HTTP status: 200
Shape: (697, 8)
  STT      Tên Thuộc tỉnh/thành phố Diện tích (km²) Dân số năm 2024 (người)  \
0   1  Ái Quốc            Hải Phòng           17,83                  24.736   
1   2  An Bình              Cần Thơ           18,39                  50.150   
2   3  An Bình            Đồng Tháp           50,07                  33.314   
3   4  An Bình              Gia Lai           73,13                  30.851   
4   5  An Biên            Hải Phòng            6,56                 116.091   

  Mật độ dân số (người/km²) Loại đô thị [ 2 ] (năm công nhận) Năm thành lập  
0                     1.387                        III (2026)          2025  
1                     2.727                        III (2026)          2025  
2                       665                         II (2026)          2025  
3                       422                        III (2026)          2025  
4                    17.697                         II (2026)          2025  


In [46]:
density_xa = density_xa[
    density_xa["Thuộc tỉnh/thành phố"].isin(
        ["Hà Nội", "Thành phố Hồ Chí Minh"]
    )
].copy()

density_xa["Thuộc tỉnh/thành phố"] = (
    density_xa["Thuộc tỉnh/thành phố"]
    .str.strip()
    .str.lower()
    .replace({
        "thành phố hồ chí minh": "hồ chí minh",
        "hà nội": "hà nội"
    })
)

density_xa["Tên"] = (
    "xã " +
    density_xa["Tên"]
    .str.strip()
    .str.lower()
)

density_xa = density_xa.drop(
    columns=["STT", "Năm thành lập"]
)

density_xa = density_xa.rename(columns={
    "Tên": "locality",
    "Thuộc tỉnh/thành phố": "region",
    "Diện tích (km²)": "locality_square",
    "Dân số năm 2025 (người)": "locality_population",
    "Mật độ dân số (người/km²)": "locality_population_density"
})

density_xa

,locality,region,locality_square,locality_population,locality_population_density
23,xã an khánh,hà nội,"28,69",102.136,3.560
31,xã an long,hồ chí minh,"100,05",17.906,179
38,xã an nhơn tây,hồ chí minh,"77,69",40.896,526
53,xã an thới đông,hồ chí minh,"257,84",18.413,71
72,xã bà điểm,hồ chí minh,"27,36",192.230,7.026
...,...,...,...,...,...
2541,xã xuân thới sơn,hồ chí minh,"35,21",96.621,2.744
2548,xã xuyên mộc,hồ chí minh,"102,96",26.917,261
2555,xã yên bài,hà nội,"68,19",21.416,314
2572,xã yên lãng,hà nội,"44,81",71.339,1.592


In [47]:
density_xa = pd.read_csv(output_path_xa)

density_xa = density_xa[
    density_xa["Thuộc tỉnh/thành phố"].isin(
        ["Hà Nội", "Thành phố Hồ Chí Minh"]
    )
].copy()

density_xa["Thuộc tỉnh/thành phố"] = (
    density_xa["Thuộc tỉnh/thành phố"]
    .str.strip()
    .str.lower()
    .replace({
        "thành phố hồ chí minh": "hồ chí minh",
        "hà nội": "hà nội"
    })
)

density_xa["Tên"] = (
    "xã " +
    density_xa["Tên"]
    .str.strip()
    .str.lower()
)

density_xa = density_xa.drop(
    columns=["Unnamed: 0", "STT", "Năm thành lập"]
)

density_xa = density_xa.rename(columns={
    "Tên": "locality",
    "Thuộc tỉnh/thành phố": "region",
    "Diện tích (km²)": "locality_square",
    "Dân số năm 2025 (người)": "locality_population",
    "Mật độ dân số (người/km²)": "locality_population_density"
})

density_xa

,locality,region,locality_square,locality_population,locality_population_density
23,xã an khánh,hà nội,"28,69",102.136,3.560
31,xã an long,hồ chí minh,"100,05",17.906,179.000
38,xã an nhơn tây,hồ chí minh,"77,69",40.896,526.000
53,xã an thới đông,hồ chí minh,"257,84",18.413,71.000
72,xã bà điểm,hồ chí minh,"27,36",192.230,7.026
...,...,...,...,...,...
2541,xã xuân thới sơn,hồ chí minh,"35,21",96.621,2.744
2548,xã xuyên mộc,hồ chí minh,"102,96",26.917,261.000
2555,xã yên bài,hà nội,"68,19",21.416,314.000
2572,xã yên lãng,hà nội,"44,81",71.339,1.592


In [48]:
density_phuong = pd.read_csv(output_path_phuong)

density_phuong = density_phuong[
    density_phuong["Thuộc tỉnh/thành phố"].isin(
        ["Hà Nội", "Thành phố Hồ Chí Minh"]
    )
].copy()

density_phuong["Thuộc tỉnh/thành phố"] = (
    density_phuong["Thuộc tỉnh/thành phố"]
    .str.strip()
    .str.lower()
    .replace({
        "thành phố hồ chí minh": "hồ chí minh",
        "hà nội": "hà nội"
    })
)

density_phuong["Tên"] = (
    "phường " +
    density_phuong["Tên"]
    .str.strip()
    .str.lower()
)

density_phuong = density_phuong.drop(
    columns=["Unnamed: 0", "STT", "Năm thành lập", "Loại đô thị [ 2 ] (năm công nhận)"]
)

density_phuong = density_phuong.rename(columns={
    "Tên": "locality",
    "Thuộc tỉnh/thành phố": "region",
    "Diện tích (km²)": "locality_square",
    "Dân số năm 2024 (người)": "locality_population",
    "Mật độ dân số (người/km²)": "locality_population_density"
})

density_phuong

,locality,region,locality_square,locality_population,locality_population_density
7,phường an đông,hồ chí minh,"1,32",81.229,61.537
11,phường an hội đông,hồ chí minh,"3,29",113.681,34.553
12,phường an hội tây,hồ chí minh,"3,81",121.004,31.760
13,phường an khánh,hồ chí minh,"15,33",76.967,5.021
16,phường an lạc,hồ chí minh,"10,47",172.134,16.441
...,...,...,...,...,...
682,phường xuân hòa,hồ chí minh,"2,22",48.464,21.831
687,phường xuân phương,hà nội,"10,81",104.947,9.708
691,phường yên hòa,hà nội,"4,10",77.029,18.788
692,phường yên nghĩa,hà nội,"13,18",49.643,3.767


In [49]:
density = pd.concat(
    [density_xa, density_phuong],
    ignore_index=True
)
density["region"].value_counts()

region
hồ chí minh    167
hà nội         126
Name: count, dtype: int64

In [60]:
density.to_csv(output_full_density, index=False)